In [1]:
import matplotlib
import numpy as np
import matplotlib.pyplot as plt
import random

print("Hello, World!")
print("NumPy version:", np.__version__)
print("Matplotlib version:", matplotlib.__version__)

Hello, World!
NumPy version: 2.2.3
Matplotlib version: 3.10.0


In [2]:
a = np.array(np.arange(10))
print(a)

[0 1 2 3 4 5 6 7 8 9]


In [3]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_prime(z):
    """Derivative of the sigmoid function."""
    return sigmoid(z)*(1-sigmoid(z))

In [20]:
class NeuralNetwork(object):
    def __init__(self, sizes):
        
        """The list ``sizes`` contains the number of neurons in the
        respective layers of the network.  For example, if the list
        was [2, 3, 1] then it would be a three-layer network, with the
        first layer containing 2 neurons, the second layer 3 neurons,
        and the third layer 1 neuron.  The biases and weights for the
        network are initialized randomly, using a Gaussian
        distribution with mean 0, and variance 1.  Note that the first
        layer is assumed to be an input layer, and by convention we
        won't set any biases for those neurons, since biases are only
        ever used in computing the outputs from later layers."""
        
        self.num_layers = len(sizes)
        self.sizes = sizes
        self.biases = [np.random.randn(y, 1) for y in sizes[1:]]
        self.weights = [np.random.randn(y, x) 
            for x, y in zip(sizes[:-1], sizes[1:])]

    def feed_forward(self, a):
        for layer_idx, (b, w) in enumerate(zip(self.biases, self.weights), start=1):
            # Here b & w are the bias vector and weight matrices for each layer
            in_size = self.sizes[layer_idx - 1]
            out_size = self.sizes[layer_idx]
            # print(f"Layer from {in_size} -> {out_size}")
            # print("Biases:", b)
            # print("Weights:", w)
            z = np.dot(w, a) + b
            a = sigmoid(z)
            # print("Activation:", a)
        return a
    
    def SGD(self, training_data, epochs, mini_batch_size, learning_rate,
            test_data=None):
        """Train the neural network using mini-batch stochastic
        gradient descent.  The ``training_data`` is a list of tuples
        ``(x, y)`` representing the training inputs and the desired
        outputs.  The other non-optional parameters are
        self-explanatory.  If ``test_data`` is provided then the
        network will be evaluated against the test data after each
        epoch, and partial progress printed out.  This is useful for
        tracking progress, but slows things down substantially."""
        n_test = len(test_data) if test_data is not None else None
        n = len(training_data)
        for j in range(epochs):
            random.shuffle(training_data)
            mini_batches = [
                training_data[k:k+mini_batch_size]
                for k in range(0, n, mini_batch_size)]
            for mini_batch in mini_batches:
                self.update_mini_batch(mini_batch, learning_rate)
            if test_data is not None:
                print(f"Epoch {j}: {self.evaluate(test_data)} / {n_test}")
            else:
                print(f"Epoch {j} complete")
                
    
    
    def update_mini_batch(self, mini_batch, learning_rate):
        """Update the network's weights and biases by applying
        gradient descent using backpropagation to a single mini batch.
        The ``mini_batch`` is a list of tuples ``(x, y)``, and ``learning rate``."""
        nabla_b = [np.zeros(b.shape) for b in self.biases]
        nabla_w = [np.zeros(w.shape) for w in self.weights]
        for x, y in mini_batch:
            delta_nabla_b, delta_nabla_w = self.back_prop(x, y)
            nabla_b = [nb+dnb for nb, dnb in zip(nabla_b, delta_nabla_b)]
            nabla_w = [nw+dnw for nw, dnw in zip(nabla_w, delta_nabla_w)]
        self.weights = [w-(learning_rate/len(mini_batch))*nw
                        for w, nw in zip(self.weights, nabla_w)]
        self.biases = [b-(learning_rate/len(mini_batch))*nb
                       for b, nb in zip(self.biases, nabla_b)]
    
    def back_prop(self, x, y):
        """Return a tuple ``(nabla_b, nabla_w)`` representing the
        gradient for the cost function C_x.  ``nabla_b`` and
        ``nabla_w`` are layer-by-layer lists of numpy arrays, similar
        to ``self.biases`` and ``self.weights``."""
        nabla_b = [np.zeros(b.shape) for b in self.biases]
        nabla_w = [np.zeros(w.shape) for w in self.weights]
        
        # feedforward
        activation = x
        activations = [x] # list to store all the activations, layer by layer
        zs = [] # list to store all the z vectors, layer by layer
        for b, w in zip(self.biases, self.weights):
            z = np.dot(w, activation)+b
            zs.append(z)
            activation = sigmoid(z)
            activations.append(activation)
        
        # backward pass
        # compute the error at the output layer
        delta = self.cost_derivative(activations[-1], y) * sigmoid_prime(zs[-1])
        nabla_b[-1] = delta
        nabla_w[-1] = np.dot(delta, activations[-2].T)
        # here l = 1 means the last layer of neurons, l = 2 is the
        # second-last layer, and so on.
        for l in range(2, self.num_layers):
            z = zs[-l]
            sp = sigmoid_prime(z)
            
            # delta is the error term for layer l+1, calculated previously
            # we use it to calculate the error term for layer l i.e. delta1
            delta1 = np.dot(self.weights[-l+1].T, delta) * sp
            
            # update delta to delta1 for next iteration
            delta = delta1
            nabla_b[-l] = delta
            nabla_w[-l] = np.dot(delta, activations[-l-1].T)
        return (nabla_b, nabla_w)
    
    def evaluate(self, test_data):
        """Return the number of test inputs for which the neural
        network outputs the correct result. Note that the neural
        network's output is assumed to be the index of whichever
        neuron in the final layer has the highest activation."""
        test_results = [(np.argmax(self.feed_forward(x)), y)
                        for (x, y) in test_data]
        return sum(int(x == y) for (x, y) in test_results)
    
    def cost_derivative(self, output_activations, y):
        """Return the vector of partial derivatives partial C_x
        partial a for the output activations."""
        return (output_activations-y)


In [21]:
import mnist_loader
training_data, validation_data, test_data = mnist_loader.load_data_wrapper()

print("Training data size:", len(training_data))
print("Validation data size:", len(validation_data))
print("Test data size:", len(test_data))

nn = NeuralNetwork([784, 16, 16, 10])

Training data size: 230000
Validation data size: 10000
Test data size: 40000


In [22]:

first_input, first_label = training_data[0]
prediction = nn.feed_forward(first_input)
pred_digit = int(np.argmax(prediction))
true_digit = mnist_loader.digit_from_vector(first_label)
cost = np.linalg.norm(prediction - first_label) ** 2
print("Predicted vector:", prediction.T)
print("True label vector:", first_label.T)
print(f"Predicted digit: {pred_digit}")
print(f"True digit: {true_digit}")
print("Cost:", cost)

b_gradients, w_gradients = nn.back_prop(first_input, first_label)
print("Bias gradients for each layer:")
for idx, b_grad in enumerate(b_gradients, start=1):
    print(f"Layer {idx} bias gradients:\n{b_grad.T}")
print("Weight gradients for each layer:")
for idx, w_grad in enumerate(w_gradients, start=1):
    print(f"Layer {idx} weight gradients:\n{w_grad}")


Predicted vector: [[0.99767559 0.02853418 0.0533999  0.51893151 0.04396884 0.81864794
  0.87558756 0.31689578 0.01521987 0.89799974]]
True label vector: [[0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]]
Predicted digit: 0
True digit: 3
Cost: 3.5762786361611343
Bias gradients for each layer:
Layer 1 bias gradients:
[[-1.40371935e-02  1.42392301e-08 -7.66654529e-04 -1.64602889e-02
   2.73737519e-04  9.02180394e-03 -5.41688526e-05  2.88706091e-02
  -1.75464625e-07 -1.26242469e-06 -4.59896808e-10 -4.12191637e-03
   7.60451395e-05  9.09071192e-03 -6.97308549e-04  1.78635543e-02]]
Layer 2 bias gradients:
[[ 0.02370114  0.00115427  0.04891519  0.04783783  0.08403815 -0.02343082
   0.06274561 -0.03737886  0.04257804 -0.02553922  0.00530918  0.05864092
  -0.00632724 -0.01851365 -0.01564874 -0.1297959 ]]
Layer 3 bias gradients:
[[ 0.00231361  0.00079097  0.00269928 -0.12009471  0.00184826  0.12153933
   0.09538124  0.06859933  0.00022812  0.08225337]]
Weight gradients for each layer:
Layer 1 weight gradients:
[